<a href="https://colab.research.google.com/github/Eng-Waheedullah-wazir/flyrank-waheed-ml-intern/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng-Waheedullah-wazir/flyrank-waheed-ml-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My baseline rule

I rank pages for review using two observed signals:

1. Staleness: older content receives a higher score because it may need a refresh review.
2. Search visibility: pages with higher `impressions_90d` receive a higher score because a refresh decision on a page with more search visibility may have more practical value.

The final score is the sum of the two signal scores.

The rule is a decision-support queue, not a prediction of future traffic or Google rankings.

### Score

- Staleness > 365 days → +3
- Staleness 181–365 days → +2
- Staleness 91–180 days → +1
- Staleness ≤90 days → +0

For search visibility:

- Top 25% of `impressions_90d` → +2
- Otherwise → +0

### Reason codes

- `STALE_HIGH_VISIBILITY` — stale and high search visibility
- `STALE` — stale but not in the high-visibility bucket
- `HIGH_VISIBILITY` — high visibility but not stale
- `NONE` — neither condition triggered

The score uses only observed/current-window fields. It does not use `trend_direction`, `trend_pct`, future windows, or the target label.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from pathlib import Path


df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')

print("Shape:", df.shape)

print("\nPossible staleness fields:")
print([c for c in df.columns if "update" in c.lower() or "stale" in c.lower()])

print("\nPossible visibility fields:")
print([c for c in df.columns if "impression" in c.lower()])


Shape: (30000, 44)

Possible staleness fields:
['days_since_last_update']

Possible visibility fields:
['impressions_90d', 'days_with_impressions', 'impressions_last_30d', 'impressions_prev_30d', 'impression_tier']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

STALENESS_COL = "days_since_last_update"
VISIBILITY_COL = "impressions_90d"

# Check that the required fields exist.
required_cols = [
    "content_id",
    "client_id",
    STALENESS_COL,
    VISIBILITY_COL,
]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# 75th percentile defines the high-visibility bucket.
visibility_threshold = df[VISIBILITY_COL].quantile(0.75)

# -----------------------------
# Staleness score
# -----------------------------
staleness_score = np.select(
    [
        df[STALENESS_COL] > 365,
        df[STALENESS_COL].between(181, 365, inclusive="both"),
        df[STALENESS_COL].between(91, 180, inclusive="both"),
    ],
    [
        3,
        2,
        1,
    ],
    default=0,
)

# -----------------------------
# Visibility score
# -----------------------------
visibility_score = np.where(
    df[VISIBILITY_COL] >= visibility_threshold,
    2,
    0,
)

# -----------------------------
# Final score
# -----------------------------
df["baseline_score"] = (
    staleness_score + visibility_score
)

# -----------------------------
# One reason code
# -----------------------------
df["reason_code"] = np.select(
    [
        (df[STALENESS_COL] > 365)
        & (df[VISIBILITY_COL] >= visibility_threshold),

        (df[STALENESS_COL] > 365),

        (df[VISIBILITY_COL] >= visibility_threshold),
    ],
    [
        "STALE_HIGH_VISIBILITY",
        "STALE",
        "HIGH_VISIBILITY",
    ],
    default="NONE",
)

# -----------------------------
# Action
# -----------------------------
df["action"] = np.where(
    df["baseline_score"] >= 3,
    "REVIEW",
    "MONITOR",
)

# -----------------------------
# Rank
# -----------------------------
queue = (
    df[
        [
            "content_id",
            "client_id",
            STALENESS_COL,
            VISIBILITY_COL,
            "baseline_score",
            "reason_code",
            "action",
        ]
    ]
    .sort_values(
        [
            "baseline_score",
            STALENESS_COL,
            VISIBILITY_COL,
            "content_id",
        ],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

queue = queue[
    [
        "rank",
        "content_id",
        "client_id",
        STALENESS_COL,
        VISIBILITY_COL,
        "baseline_score",
        "reason_code",
        "action",
    ]
]

print(queue.head(20).to_string(index=False))



OUTPUT_PATH = Path("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv")

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

queue.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Rows written: {len(queue):,}")
print(f"Output: {OUTPUT_PATH}")


 rank           content_id         client_id  days_since_last_update  impressions_90d  baseline_score     reason_code action
    1 content_cf56e2e2e282 client_7f2253d7e2                     194            61678               4 HIGH_VISIBILITY REVIEW
    2 content_7368877ea310 client_7f2253d7e2                     194            59472               4 HIGH_VISIBILITY REVIEW
    3 content_1bfaa38ff26c client_7f2253d7e2                     194            25715               4 HIGH_VISIBILITY REVIEW
    4 content_5feee3994adb client_7f2253d7e2                     194             7812               4 HIGH_VISIBILITY REVIEW
    5 content_b16bd7307b39 client_7f2253d7e2                     194             4590               4 HIGH_VISIBILITY REVIEW
    6 content_fe16a55cd13d client_7f2253d7e2                     194             4556               4 HIGH_VISIBILITY REVIEW
    7 content_ecb6215e79fd client_7f2253d7e2                     194             4429               4 HIGH_VISIBILITY REVIEW


URLError: <urlopen error no host given>

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


The following review treats the ranked queue as decision-support.

For each selected page I record:

- the action produced by the rule,
- the reason code,
- a confidence note based only on the observed signals,
- what additional information could make the pick wrong.

A high score means the rule considers the page worth reviewing first. It does not establish that the page actually needs a content refresh.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20).copy()

def confidence_note(row):
    score = row["baseline_score"]

    if score >= 5:
        return "Higher rule confidence because both signals contribute strongly."
    elif score >= 3:
        return "Moderate rule confidence because the page meets the review threshold."
    else:
        return "Lower rule confidence because only a weaker signal contributes."

def wrong_if_note(row):
    if row["reason_code"] == "STALE_HIGH_VISIBILITY":
        return (
            "Could be wrong if the update date is stale/inaccurate "
            "or the high impressions do not represent useful current demand."
        )
    elif row["reason_code"] == "STALE":
        return (
            "Could be wrong if the content is intentionally unchanged "
            "or the recorded update date is inaccurate."
        )
    elif row["reason_code"] == "HIGH_VISIBILITY":
        return (
            "Could be wrong if high impressions do not translate into "
            "a meaningful refresh opportunity."
        )
    else:
        return (
            "Could be wrong because the rule has weak evidence "
            "for prioritizing this page."
        )

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1,
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_if_note,
    axis=1,
)

review_cols = [
    "rank",
    "action",
    "reason_code",
    "baseline_score",
    "confidence_note",
    "what_would_make_it_wrong",
]

print(
    top20[review_cols].to_string(index=False)
)


 rank action     reason_code  baseline_score                                                       confidence_note                                                                            what_would_make_it_wrong
    1 REVIEW HIGH_VISIBILITY               4 Moderate rule confidence because the page meets the review threshold.          Could be wrong if high impressions do not translate into a meaningful refresh opportunity.
    2 REVIEW HIGH_VISIBILITY               4 Moderate rule confidence because the page meets the review threshold.          Could be wrong if high impressions do not translate into a meaningful refresh opportunity.
    3 REVIEW HIGH_VISIBILITY               4 Moderate rule confidence because the page meets the review threshold.          Could be wrong if high impressions do not translate into a meaningful refresh opportunity.
    4 REVIEW HIGH_VISIBILITY               4 Moderate rule confidence because the page meets the review threshold.          Could be wrong i

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks

The weakest picks are rows that receive a review action even though the rule has limited evidence, or rows whose observed signals may not translate into an actual refresh opportunity.

The rule is intentionally simple. A high score is therefore not proof that a page should be refreshed.

## Leakage check

The baseline does not use:

- `trend_direction`
- `trend_pct`
- `is_declining_label`
- future-window measurements
- product flags or downstream model outputs

The score is constructed only from current/observed staleness and search-visibility signals.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the lowest-scoring rows that still received REVIEW.
weak_picks = (
    queue[queue["action"] == "REVIEW"]
    .sort_values(
        ["baseline_score", "rank"],
        ascending=[True, True],
    )
    .head(10)
)

print("Weakest REVIEW picks:")
print(
    weak_picks.to_string(index=False)
)


# Explicit leakage checks.

LEAKAGE_FIELDS = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

used_for_score = {
    STALENESS_COL,
    VISIBILITY_COL,
}

print("Fields used by baseline:")
print(sorted(used_for_score))

print("\nLeakage fields:")
print(sorted(LEAKAGE_FIELDS))

# None of the leakage fields may be used in the score.
assert used_for_score.isdisjoint(LEAKAGE_FIELDS)

# Confirm the target is not accidentally present in the queue
# as an input to the scoring calculation.
score_columns = {
    STALENESS_COL,
    VISIBILITY_COL,
    "baseline_score",
    "reason_code",
    "action",
}

assert "trend_direction" not in score_columns
assert "trend_pct" not in score_columns
assert "is_declining_label" not in score_columns

print("\nLeakage check: PASSED")


Weakest REVIEW picks:
 rank           content_id         client_id  days_since_last_update  impressions_90d  baseline_score     reason_code action
   10 content_55a5b1c46474 client_4ec9599fc2                     373               35               3           STALE REVIEW
   11 content_f6fdf87348f6 client_4ec9599fc2                     373                2               3           STALE REVIEW
   12 content_3f3576c295f5 client_4ec9599fc2                     373                1               3           STALE REVIEW
   13 content_1b4ec72dafd4 client_4ec9599fc2                     372                2               3           STALE REVIEW
   14 content_8d56efff1e71 client_4ec9599fc2                     372                1               3           STALE REVIEW
   15 content_cb7e312f5d32 client_9f14025af0                     151            21272               3 HIGH_VISIBILITY REVIEW
   16 content_a5dbb404bdc2 client_f369cb89fc                     106            79035               3 H

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.